In [1]:
import pandas as pd
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("SMSSpamCollection", sep='\t', names=["label", "message"])

# Encode labels
le = LabelEncoder()
df['label_num'] = le.fit_transform(df['label'])  # ham=0, spam=1

# Features and labels
X = df['message'].values
y = df['label_num'].values

# Tokenization
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X)
X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=100, padding='post')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42)

# Build model
model = Sequential([
    Embedding(input_dim=5000, output_dim=16, input_length=100),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile and train
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

# Save model
model.save("sms_model.h5")

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved successfully.")


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8301 - loss: 0.4535 - val_accuracy: 0.8664 - val_loss: 0.3650
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8664 - loss: 0.3631 - val_accuracy: 0.8664 - val_loss: 0.3586
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8606 - loss: 0.3676 - val_accuracy: 0.8664 - val_loss: 0.3497
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8628 - loss: 0.3534 - val_accuracy: 0.8664 - val_loss: 0.3351
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8645 - loss: 0.3319 - val_accuracy: 0.8664 - val_loss: 0.3105


Model and tokenizer saved successfully.


In [2]:
import tensorflow as tf
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Load the trained model
model = tf.keras.models.load_model("sms_model.h5")

# Preprocess SMS input
def preprocess_sms(sms_text):
    seq = tokenizer.texts_to_sequences([sms_text])
    padded = pad_sequences(seq, maxlen=100, padding='post')
    return padded

# Predict function
def predict_sms(sms_text):
    processed = preprocess_sms(sms_text)
    prediction = model.predict(processed)[0][0]
    label = "SPAM" if prediction > 0.5 else "HAM"
    confidence = prediction if prediction > 0.5 else 1 - prediction
    return label, round(confidence, 2)

# Main
if __name__ == "__main__":
    sms_input = input("📩 Enter an SMS to classify: ")
    label, confidence = predict_sms(sms_input)
    print(f" Prediction: {label} (Confidence: {confidence * 100:.1f}%)")


📩 Enter an SMS to classify: Win Lottery Guaranteed
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
 Prediction: HAM (Confidence: 93.0%)
